# 02 · EfficientNet-B0 transfer learning

ImageNet-pretrained EfficientNet-B0 (torchvision) fine-tuned for 3-class substructure classification.

**Single-channel adaptation.** The pretrained stem conv has weights of shape (32, 3, 3, 3).
We replace it with a (32, 1, 3, 3) conv whose kernel is the **sum** over RGB. For a
grayscale image *x*, the new layer responds exactly as the original did to the RGB image
(*x*, *x*, *x*), so every downstream pretrained feature is preserved. (Averaging, as in the
earlier version, scales the activations by ⅓ and mismatches the pretrained BatchNorm statistics.)

**Two-stage fine-tuning.**
1. Epoch 1: backbone frozen, only the new head trains (avoids wrecking pretrained features with
   gradients from a random head).
2. Then the whole network trains with **discriminative learning rates**: head 1e-3, backbone 3e-4.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplense import CLASS_NAMES
from deeplense.utils import get_device, seed_everything

seed_everything(42)
DEVICE = get_device()
DATA_ROOT = ROOT / "data" / "lensing"
RESULTS = ROOT / "results"
print("device:", DEVICE)

In [ ]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# SMOKE = True : a few hundred images, 2 epochs, just to check everything runs (laptop).
# SMOKE = False: full dataset and full recipe (GPU recommended).
# If a full run already exists in results/<run>/ (e.g. from scripts/train.py),
# it is loaded instead of retraining unless RETRAIN = True.
SMOKE = True
RETRAIN = False

In [ ]:
from deeplense.models import EfficientNetClassifier
from deeplense.models.common import rgb_to_single_channel
import torchvision.models as tvm

rgb = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1).features[0][0].weight
print("pretrained stem:", tuple(rgb.shape), "-> adapted:", tuple(rgb_to_single_channel(rgb).shape))

# Equivalence check: gray conv on x == RGB conv on (x, x, x)
x = torch.rand(1, 1, 32, 32)
a = torch.nn.functional.conv2d(x.repeat(1, 3, 1, 1), rgb, padding=1)
b = torch.nn.functional.conv2d(x, rgb_to_single_channel(rgb), padding=1)
print("max |difference|:", (a - b).abs().max().item())

## Train (or load)

In [ ]:
from dataclasses import replace
from deeplense.data import DataConfig, build_loaders
from deeplense.metrics import predict
from deeplense.train import fit
from deeplense.utils import load_json
sys.path.insert(0, str(ROOT / "scripts"))
from train import RECIPES

cfg = RECIPES["efficientnet"]
data_cfg = DataConfig(root=str(DATA_ROOT), num_workers=2)
if SMOKE:
    cfg = replace(cfg, epochs=2)
    data_cfg = replace(data_cfg, train_per_class=300, test_per_class=100)
RUN = "efficientnet" + ("_smoke" if SMOKE else "")

loaders = build_loaders(data_cfg)
model = EfficientNetClassifier(pretrained=True)
criterion = None
run_dir = RESULTS / RUN

if (run_dir / "best.pt").exists() and not RETRAIN:
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu"))
    model.to(DEVICE)
    results = load_json(run_dir / "metrics.json")
    history = load_json(run_dir / "history.json")
    probs, labels = predict(model, loaders["test"], DEVICE)
    print(f"Loaded {run_dir}  (best epoch {results['best_epoch']})")
else:
    model, out = fit(model, loaders, cfg, DEVICE, criterion=criterion, out_dir=RESULTS, run_name=RUN)
    results, history, probs, labels = out, out["history"], out["probs"], out["labels"]

## Evaluation on the held-out test set

In [ ]:
from deeplense.metrics import classification_metrics, plot_confusion, plot_history, plot_roc

plot_history(history, title=RUN); plt.show()

m = classification_metrics(probs, labels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_roc(probs, labels, title=RUN, ax=axes[0])
plot_confusion(m["confusion_matrix"], title="Test confusion matrix", ax=axes[1])
plt.tight_layout(); plt.savefig(RESULTS / RUN / "roc_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

print(f"Test accuracy {m['accuracy']:.4f} | macro AUC {m['macro_auc']:.4f}")
for k, v in m["auc_per_class"].items():
    print(f"  {k:<20} AUC {v:.4f}")